In [25]:
import pandas as pd 
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

def select_top_variable_features(df: pd.DataFrame, k: int = 50) -> pd.DataFrame:
    numeric_df = df.select_dtypes(include=["number"]).copy()
    if numeric_df.shape[1] == 0:
        raise ValueError("No numeric columns found to plot/normalize.")
    if k is None or k <= 0 or numeric_df.shape[1] <= k:
        return numeric_df
    variances = numeric_df.var(axis=0, ddof=0).sort_values(ascending=False)
    cols = variances.head(k).index
    return numeric_df.loc[:, cols]

def normalize_df(
    df: pd.DataFrame,
    method: str = "zscore",
    eps: float = 1e-8,
) -> tuple[pd.DataFrame, dict]:
    """Normalize numeric columns of a DataFrame and return (normalized_df, params).

    Supported methods:
      - 'zscore': (x - mean) / std
      - 'minmax': (x - min) / (max - min)

    The returned params can be passed into `unnormalize_df` to recover original values.
    """
    numeric_df = df.select_dtypes(include=["number"]).copy()
    if numeric_df.shape[1] == 0:
        raise ValueError("No numeric columns found to normalize.")

    method = method.lower().strip()
    if method == "zscore":
        mean = numeric_df.mean(axis=0)
        std = numeric_df.std(axis=0, ddof=0).replace(0, np.nan)
        std = std.fillna(1.0)
        normalized = (numeric_df - mean) / (std + eps)
        params = {"method": method, "mean": mean, "std": std, "eps": eps}
        return normalized, params

    if method == "minmax":
        min_v = numeric_df.min(axis=0)
        max_v = numeric_df.max(axis=0)
        denom = (max_v - min_v).replace(0, np.nan).fillna(1.0)
        normalized = (numeric_df - min_v) / (denom + eps)
        params = {"method": method, "min": min_v, "max": max_v, "eps": eps}
        return normalized, params

    raise ValueError(f"Unsupported normalization method: {method}. Use 'zscore' or 'minmax'.")

def unnormalize_df(df_normalized: pd.DataFrame, params: dict) -> pd.DataFrame:
    method = str(params.get("method", "")).lower().strip()
    eps = float(params.get("eps", 1e-8))

    numeric_df = df_normalized.select_dtypes(include=["number"]).copy()
    if numeric_df.shape[1] == 0:
        raise ValueError("No numeric columns found to unnormalize.")

    if method == "zscore":
        mean = params["mean"]
        std = params["std"]
        return numeric_df * (std + eps) + mean

    if method == "minmax":
        min_v = params["min"]
        max_v = params["max"]
        denom = (max_v - min_v).replace(0, np.nan).fillna(1.0)
        return numeric_df * (denom + eps) + min_v

    raise ValueError(f"Unsupported normalization method: {method}.")

def save_boxplot(
    df: pd.DataFrame,
    output_path: str,
    title: str | None = None,
    figsize: tuple[int, int] = (18, 6),
    rotate_xticks: int = 0,
    showfliers: bool = True,
) -> None:
    plot_df = df.select_dtypes(include=["number"]).copy()
    if plot_df.shape[1] == 0:
        raise ValueError("No numeric columns found to plot.")
    fig, ax = plt.subplots(figsize=figsize)
    sns.boxplot(data=plot_df, ax=ax, showfliers=showfliers)
    if title:
        ax.set_title(title)
    ax.tick_params(axis="x", labelrotation=rotate_xticks)
    fig.tight_layout()
    fig.savefig(output_path, dpi=150, bbox_inches="tight")
    plt.close(fig)

def save_boxplot_compare(
    raw_df: pd.DataFrame,
    normalized_df: pd.DataFrame,
    output_path: str,
    suptitle: str = "Gene Expression Distribution",
    figsize: tuple[int, int] = (22, 6),
    rotate_xticks: int = 0,
    showfliers: bool = True,
) -> None:
    raw_plot = raw_df.select_dtypes(include=["number"]).copy()
    norm_plot = normalized_df.select_dtypes(include=["number"]).copy()
    if raw_plot.shape[1] == 0 or norm_plot.shape[1] == 0:
        raise ValueError("No numeric columns found to plot.")

    fig, axes = plt.subplots(1, 2, figsize=figsize)
    sns.boxplot(data=raw_plot, ax=axes[0], showfliers=showfliers)
    axes[0].set_title("Raw")
    sns.boxplot(data=norm_plot, ax=axes[1], showfliers=showfliers)
    axes[1].set_title("Normalized")

    for ax in axes:
        ax.tick_params(axis="x", labelrotation=rotate_xticks)

    fig.suptitle(suptitle)
    fig.tight_layout(rect=[0, 0, 1, 0.95])
    fig.savefig(output_path, dpi=150, bbox_inches="tight")
    plt.close(fig)


In [26]:
hall_mark_file_path = "/home/licongcong/Desktop/TCGA/tcga-gbm/Hallmark/hallmark_pathways.csv"
reactome_file_path = "/home/licongcong/Desktop/TCGA/tcga-gbm/Reactome/reactome_pathways.csv"
clincal_file_path = "/home/licongcong/Desktop/TCGA/tcga-gbm/TCGA_GBM_clinicalMatrix.csv"
genomic_file_path = "/home/licongcong/Desktop/TCGA/tcga-gbm/genomic_data_clean.csv"

In [27]:
# analysis for hallmark pathways
hallmark_df = pd.read_csv(hall_mark_file_path, header=None)
hallmark_df.rename(columns={0: 'pathway', 1:'geneset'}, inplace=True)

print("Hallmark Pathways DataFrame:")
print(hallmark_df.shape)
print(hallmark_df.head())

Hallmark Pathways DataFrame:
(50, 2)
                        pathway  \
0         HALLMARK_ADIPOGENESIS   
1  HALLMARK_ALLOGRAFT_REJECTION   
2    HALLMARK_ANDROGEN_RESPONSE   
3         HALLMARK_ANGIOGENESIS   
4      HALLMARK_APICAL_JUNCTION   

                                             geneset  
0  ABCA1,ABCB8,ACAA2,ACADL,ACADM,ACADS,ACLY,ACO2,...  
1  AARS1,ABCE1,ABI1,ACHE,ACVR2A,AKT1,APBB1,B2M,BC...  
2  ABCC4,ABHD2,ACSL3,ACTN1,ADAMTS1,ADRM1,AKAP12,A...  
3  APOH,APP,CCND2,COL3A1,COL5A2,CXCL6,FGFR1,FSTL1...  
4  ACTA1,ACTB,ACTC1,ACTG1,ACTG2,ACTN1,ACTN2,ACTN3...  


In [ ]:
def plot_pathway_membership_per_gene(df_matrix, top_n=30):
    fig, ax = plt.subplots(figsize=(14, 6))
    fig.patch.set_facecolor('#0D1F3C')
    ax.set_facecolor('#1A3558')

    # How many pathways each gene belongs to
    gene_counts = df_matrix.sum(axis=1).sort_values(ascending=False)
    top_genes   = gene_counts.head(top_n)

    colors = ['#EF4444' if c >= 5 else
              '#F59E0B' if c >= 3 else
              '#0D9488'
              for c in top_genes.values]

    bars = ax.bar(range(len(top_genes)),
                  top_genes.values,
                  color=colors, alpha=0.9)

    # Value labels
    for bar, val in zip(bars, top_genes.values):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.05,
                str(int(val)),
                ha='center', va='bottom',
                fontsize=9, color='white')

    ax.set_xticks(range(len(top_genes)))
    ax.set_xticklabels(top_genes.index,
                       rotation=45, ha='right',
                       fontsize=9, color='white')
    ax.set_ylabel("Number of Pathways",
                  color='white', fontsize=11)
    ax.set_title(
        f"Top {top_n} Genes by Pathway Membership\n"
        f"(How many Hallmark pathways each gene belongs to)",
        fontsize=13, fontweight='bold',
        color='white', pad=12
    )
    ax.tick_params(colors='white')

    # Legend
    legend_elements = [
        mpatches.Patch(color='#EF4444', label='≥5 pathways (Hub gene)'),
        mpatches.Patch(color='#F59E0B', label='3–4 pathways'),
        mpatches.Patch(color='#0D9488', label='1–2 pathways'),
    ]
    ax.legend(handles=legend_elements,
              facecolor='#0D1F3C',
              labelcolor='white', fontsize=9)

    for spine in ax.spines.values():
        spine.set_edgecolor('#0D9488')

    plt.tight_layout()
    # plt.savefig('fig3_gene_pathway_membership.png',
    #             dpi=150, bbox_inches='tight',
    #             facecolor='#0D1F3C')
    plt.show()
    # print("Saved: fig3_gene_pathway_membership.png")

# plot_pathway_membership_per_gene(df_matrix, top_n=30)

In [29]:
# build the pathway-gene mapping dictionary
pathway_gene_dict = {}
for _, row in hallmark_df.iterrows():
    pathway = row['pathway']
    genes = str(row["geneset"]).split(",")
    pathway_gene_dict[pathway] = genes
    print(f"Pathway: {pathway}, Number of genes: {len(genes)}")

all_genes = set()
for genes in pathway_gene_dict.values():
    all_genes.update(genes)
print(f"Total unique genes in hallmark pathways: {len(all_genes)}")

df_matrix = pd.DataFrame(0, index=sorted(all_genes), columns=pathway_gene_dict.keys())
df_matrix.rename(columns={df_matrix.columns[-1]: 'gene_name'}, inplace=True)
print()

# Step 5: Fill matrix
for pathway, genes in pathway_gene_dict.items():
    for gene in genes:
        if gene in df_matrix.index:
            df_matrix.loc[gene, pathway] = 1

df_matrix = df_matrix.reset_index()
df_matrix.rename(columns={'index': 'gene_name'}, inplace=True)

# Step 6: Save to CSV
df_matrix.to_csv("/home/licongcong/Desktop/TCGA/tcga-gbm/Hallmark/gene_pathway_matrix.csv", index=False)

Pathway: HALLMARK_ADIPOGENESIS, Number of genes: 200
Pathway: HALLMARK_ALLOGRAFT_REJECTION, Number of genes: 200
Pathway: HALLMARK_ANDROGEN_RESPONSE, Number of genes: 101
Pathway: HALLMARK_ANGIOGENESIS, Number of genes: 36
Pathway: HALLMARK_APICAL_JUNCTION, Number of genes: 200
Pathway: HALLMARK_APICAL_SURFACE, Number of genes: 44
Pathway: HALLMARK_APOPTOSIS, Number of genes: 161
Pathway: HALLMARK_BILE_ACID_METABOLISM, Number of genes: 112
Pathway: HALLMARK_CHOLESTEROL_HOMEOSTASIS, Number of genes: 74
Pathway: HALLMARK_COAGULATION, Number of genes: 138
Pathway: HALLMARK_COMPLEMENT, Number of genes: 200
Pathway: HALLMARK_DNA_REPAIR, Number of genes: 150
Pathway: HALLMARK_E2F_TARGETS, Number of genes: 200
Pathway: HALLMARK_EPITHELIAL_MESENCHYMAL_TRANSITION, Number of genes: 200
Pathway: HALLMARK_ESTROGEN_RESPONSE_EARLY, Number of genes: 200
Pathway: HALLMARK_ESTROGEN_RESPONSE_LATE, Number of genes: 200
Pathway: HALLMARK_FATTY_ACID_METABOLISM, Number of genes: 158
Pathway: HALLMARK_G2M_CH

In [30]:
# analysis for genomic data of hallmark pathways
genomic_df = pd.read_csv(genomic_file_path, header=None)
genomic_df.rename(columns={0: 'gene_names', 1:'patient_id', 2:'expression'}, inplace=True)

# print("Genomic DataFrame:")
print(genomic_df.shape)
print()
print(genomic_df.head())

hallmark_df = pd.read_csv("/home/licongcong/Desktop/TCGA/tcga-gbm/Hallmark/gene_pathway_matrix.csv")
hallmark_gene_df = hallmark_df.iloc[:, [0]]
print(hallmark_gene_df.head())
print(f"Hallmark Pathway Matrix DataFrame:{hallmark_gene_df.head()}")

(17412111, 3)

  gene_names    patient_id  expression
0  5_8S_rRNA  TCGA-02-0003           0
1  5_8S_rRNA  TCGA-02-0016           0
2  5_8S_rRNA  TCGA-02-0026           0
3  5_8S_rRNA  TCGA-02-0033           0
4  5_8S_rRNA  TCGA-02-0038           0
  gene_name
0       A2M
1      AAAS
2     AADAT
3     AARS1
4      ABAT
Hallmark Pathway Matrix DataFrame:  gene_name
0       A2M
1      AAAS
2     AADAT
3     AARS1
4      ABAT


In [31]:
print(set(genomic_df["gene_names"]) & set(hallmark_gene_df["gene_name"]))

{'IL7R', 'SDHC', 'IL18BP', 'MERTK', 'CAB39', 'DCAF10', 'ARL8A', 'PTBP2', 'ATP6V1E1', 'AOX1', 'SWAP70', 'INSM1', 'LTBP1', 'MKRN1', 'TSPO2', 'GZMA', 'ARID5B', 'RAP1GAP', 'PLVAP', 'PPOX', 'IFNGR1', 'ID3', 'MDH2', 'ASF1A', 'CPB1', 'EREG', 'NUMB', 'HDAC3', 'PFKFB1', 'ELOVL5', 'ANP32E', 'PMM2', 'PDCD4', 'OLR1', 'ADCY1', 'PCGF5', 'ITGA7', 'ARFGAP3', 'SLC20A1', 'EPB41L2', 'PML', 'ARL6IP1', 'FN1', 'KIF13B', 'KIF15', 'SEC31A', 'PEBP1', 'SELENOW', 'SIK1', 'YKT6', 'BCL2L10', 'SLC35B2', 'CA6', 'PIAS1', 'NDUFV1', 'SNX10', 'SLC23A2', 'TRAT1', 'UQCR11', 'AADAT', 'PCSK2', 'ELOVL2', 'ACVR1B', 'MAPK14', 'TGFBR1', 'HLA-DOA', 'CAMK4', 'KIF11', 'DNM1L', 'IGF1', 'GCDH', 'COX15', 'LAMB3', 'MYO1E', 'COL4A1', 'RBSN', 'TMEM140', 'EHD1', 'MRPL40', 'RHCE', 'DDAH1', 'POLR2K', 'COPZ2', 'PTK2B', 'SIRT6', 'CYP46A1', 'EHHADH', 'NDRG1', 'ALDH3A1', 'GPX2', 'DGUOK', 'TCP11', 'UPB1', 'LTA', 'CXCL11', 'PRDX1', 'AGO2', 'IER3', 'SGCB', 'HDGF', 'IL1R2', 'BATF', 'GMPR2', 'SOS1', 'SOCS3', 'PKP3', 'INSL5', 'TF', 'CDC45', 'STAG1',

In [32]:
# Filter
df_filter = genomic_df[
    genomic_df["gene_names"].isin(hallmark_gene_df["gene_name"])
]

print(df_filter.head())
print("Filtered shape:", df_filter.shape)

     gene_names    patient_id  expression
1758        A2M  TCGA-02-0003         258
1759        A2M  TCGA-02-0016         189
1760        A2M  TCGA-02-0026          91
1761        A2M  TCGA-02-0033        2534
1762        A2M  TCGA-02-0038         266
Filtered shape: (1280703, 3)


In [33]:
df_pivot = df_filter.pivot_table(
    index="patient_id",
    columns="gene_names",
    values="expression",
    aggfunc="sum",
    fill_value=0
)
df_pivot.to_csv("/home/licongcong/Desktop/TCGA/tcga-gbm/genomic_clean_pivot.csv")

In [34]:
df_pivot = pd.read_csv("/home/licongcong/Desktop/TCGA/tcga-gbm/genomic_clean_pivot.csv")
print(df_pivot.head(), df_pivot.columns)

     patient_id   A2M  AAAS  AADAT  AARS1  ABAT  ABCA1  ABCA2  ABCA3  ABCA4  \
0  TCGA-02-0003   258    11      5     24    50     23     15     18      0   
1  TCGA-02-0016   189    12      2     26    40     35     12     25      0   
2  TCGA-02-0026    91    15      3     15    27     91     31     21      0   
3  TCGA-02-0033  2534    10      1     29     5    109     12     13      0   
4  TCGA-02-0038   266    12      1     23    37     45     22     26      0   

   ...  ZNF292  ZNF365  ZNF639  ZNF707  ZNFX1  ZNRF4  ZPBP  ZW10  ZWINT  ZYX  
0  ...       7       2       7       1     11      0     0    10     10   21  
1  ...       8       2       6       2     22      0     0     9      4   36  
2  ...      14       2       5       2     11      0     0    13     37   16  
3  ...       5       1       4       2     28      0     0     9     13   54  
4  ...       4       1       4       1     24      0     0     6      7   36  

[5 rows x 4372 columns] Index(['patient_id', 'A2M'

In [ ]:
# Subplot (raw vs normalized) similar to the reference figure
feature_df = df_pivot.drop(columns=["patient_id"], errors="ignore")

# Plot only ~200 genes for readability (top variable genes).
plot_df = select_top_variable_features(feature_df, k=200)

normalized_df, norm_params = normalize_df(plot_df, method="zscore")
# save_boxplot_compare(
#     plot_df,
#     normalized_df,
#     output_path="mle_segmentation_result.png",
#     suptitle="Gene Expression Distribution",
#     rotate_xticks=0,
#     showfliers=True,
# )
# print("Saved: mle_segmentation_result.png")

restored_df = unnormalize_df(normalized_df, norm_params)
max_abs_diff = (restored_df - plot_df).abs().to_numpy().max()
print(f"Unnormalize check (max abs diff): {max_abs_diff:.6g}")


Saved: mle_segmentation_result.png
Unnormalize check (max abs diff): 3.63798e-12
